CARGA DE BRONZE Y REGISTRO POR PROCESO

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "compras_rd"

bronze = spark.table(f"{catalogo}.bronze.contratos_raw")
print("Filas en Bronze:", bronze.count())
print("ocid únicos:", bronze.select("ocid").distinct().count())

# Si un ocid se repite, conserva el registro más reciente
w = Window.partitionBy("ocid").orderBy(F.col("date").desc())
base = (bronze.withColumn("_rn", F.row_number().over(w))
        .where("_rn = 1").drop("_rn"))
print("Filas tras deduplicar:", base.count())

def guardar(df, nombre):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{catalogo}.silver.{nombre}"))
    print(f"silver.{nombre} OK:", spark.table(f"{catalogo}.silver.{nombre}").count())

EXPLORACION DE LOS VALORES

In [0]:
%python
display(base.groupBy("tender.procurementMethod", "tender.procurementMethodDetails")
        .count().orderBy(F.desc("count")))

display(base.groupBy("tender.status").count().orderBy(F.desc("count")))
display(base.groupBy("tender.value.currency").count())

display(base.select(F.explode("parties").alias("p"))
        .select(F.explode("p.roles").alias("rol"))
        .groupBy("rol").count())

In [0]:
%python
procesos = (
    base.select(
        "ocid",
        F.try_to_timestamp(F.col("date")).alias("fecha_publicacion"),
        F.col("tender.id").alias("tender_id"),
        F.col("tender.title").alias("titulo"),
        F.col("tender.procuringEntity.id").alias("institucion_id"),
        F.trim(F.col("tender.procuringEntity.name")).alias("institucion"),
        F.col("tender.procurementMethod").alias("modalidad"),
        F.col("tender.procurementMethodDetails").alias("modalidad_detalle"),
        F.col("tender.mainProcurementCategory").alias("categoria"),
        F.col("tender.status").alias("estado"),
        F.col("tender.value.amount").alias("monto_estimado"),
        F.col("tender.value.currency").alias("moneda"),
        F.col("tender.numberOfTenderers").alias("num_oferentes"),
        F.try_to_timestamp(F.col("tender.tenderPeriod.startDate")).alias("inicio_oferta"),
        F.try_to_timestamp(F.col("tender.tenderPeriod.endDate")).alias("fin_oferta"),
        F.try_to_timestamp(F.col("tender.awardPeriod.startDate")).alias("inicio_adjudicacion"),
        F.try_to_timestamp(F.col("tender.awardPeriod.endDate")).alias("fin_adjudicacion"),
        F.col("planning.budget.amount.amount").alias("presupuesto"),
    )
    .withColumn("anio", F.year("fecha_publicacion"))
    .withColumn("mes", F.month("fecha_publicacion"))
    .withColumn("flag_monto_invalido",
                F.col("monto_estimado").isNull() | (F.col("monto_estimado") <= 0))
)

guardar(procesos, "procesos")

In [0]:
%python
adjudicaciones = (
    base.select("ocid", F.explode("awards").alias("a"))
    .select(
        "ocid",
        F.col("a.id").alias("award_id"),
        F.try_to_timestamp(F.col("a.date")).alias("fecha_adjudicacion"),
        F.col("a.status").alias("estado_adjudicacion"),
        F.size("a.suppliers").alias("num_proveedores"),
        F.expr("try_element_at(a.suppliers, 1).id").alias("proveedor_id"),
        F.expr("try_element_at(a.suppliers, 1).name").alias("proveedor"),
        F.expr("""aggregate(a.items, CAST(0 AS DOUBLE),
                  (acc, x) -> acc + coalesce(x.quantity, 0) * coalesce(x.unit.value.amount, 0))""")
         .alias("monto_adjudicado"),
        F.expr("try_element_at(a.items, 1).unit.value.currency").alias("moneda"),
    )
)

guardar(adjudicaciones, "adjudicaciones")

In [0]:
%python
contratos = (
    base.select("ocid", F.explode("contracts").alias("c"))
    .select(
        "ocid",
        F.col("c.id").alias("contrato_id"),
        F.col("c.awardID").alias("award_id"),
        F.col("c.status").alias("estado_contrato"),
        F.col("c.title").alias("titulo_contrato"),
        F.col("c.value.amount").alias("monto_contrato"),
        F.col("c.value.currency").alias("moneda"),
        F.try_to_timestamp(F.col("c.period.startDate")).alias("inicio_contrato"),
        F.try_to_timestamp(F.col("c.period.endDate")).alias("fin_contrato"),
    )
    .withColumn("flag_monto_invalido",
                F.col("monto_contrato").isNull() | (F.col("monto_contrato") <= 0))
    .withColumn("flag_fecha_incoherente",
                F.coalesce(F.col("inicio_contrato") > F.col("fin_contrato"), F.lit(False)))
)

guardar(contratos, "contratos")

In [0]:
%python
proveedores = (
    base.select("ocid", F.explode("parties").alias("p"))
    .where(F.array_contains("p.roles", "supplier"))
    .select(
        F.col("p.id").alias("party_id"),
        F.col("p.identifier.id").alias("rnc"),
        F.trim(F.col("p.name")).alias("nombre"),
        F.col("p.details.scale").alias("escala"),
        F.col("p.details.gender").alias("genero"),
        F.col("p.address.region").alias("region"),
    )
    .dropDuplicates(["party_id"])
)

guardar(proveedores, "proveedores")

In [0]:
%python
catalogo = "compras_rd"
display(spark.sql(f"""
SELECT count(*) AS total,
       sum(CASE WHEN fecha_publicacion IS NULL THEN 1 ELSE 0 END) AS sin_fecha,
       sum(CASE WHEN flag_monto_invalido THEN 1 ELSE 0 END) AS monto_invalido
FROM {catalogo}.silver.procesos"""))

display(spark.sql(f"""
SELECT num_proveedores, count(*) AS n
FROM {catalogo}.silver.adjudicaciones GROUP BY 1 ORDER BY 1"""))

display(spark.sql(f"""
SELECT count(*) AS contratos,
       sum(CASE WHEN flag_monto_invalido THEN 1 ELSE 0 END) AS monto_invalido,
       sum(CASE WHEN flag_fecha_incoherente THEN 1 ELSE 0 END) AS fechas_incoherentes
FROM {catalogo}.silver.contratos"""))

display(spark.sql(f"""
SELECT count(*) AS pares,
       sum(CASE WHEN abs(a.monto_adjudicado - c.monto_contrato) < 1 THEN 1 ELSE 0 END) AS coinciden
FROM {catalogo}.silver.adjudicaciones a
JOIN {catalogo}.silver.contratos c
  ON a.ocid = c.ocid AND a.award_id = c.award_id"""))